## Softmax, Numerically Stable

1. Raw softmax $e^{z_i}/\sum e^{z_j}$ overflows for large logits (e.g. $z=1000$). Fix: subtract $\max(z)$ from every logit first — $e^{z-\max(z)}/\sum e^{z-\max(z)}$ gives the identical result (the max cancels in the ratio) but every exponent is now $\le0$, so no overflow.
2. `axis=1` = across columns, i.e. per row (do this per sample, over its classes). `axis=0` = down rows, i.e. per column.
3. `keepdims=True` preserves the reduced dimension as size-1 (shape `(n,1)` not `(n,)`) so it broadcasts cleanly against the original array; `.squeeze()` drops that dimension back out when a flat 1D result is wanted.


In [ ]:
import numpy as np

def softmax(z: np.ndarray) -> np.ndarray:
    """
    Computes a numerically stable softmax function.
    Args:
        z (np.ndarray): A 1D or 2D array of raw scores (logits).
    Returns:
        np.ndarray: The probability distribution.
    """
    # Ensure z is at least 2D for consistent max operation
    if z.ndim == 1:
        z = z.reshape(1, -1)

    # The stability trick: subtract the max for each sample
    max_z = np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z - max_z)

    # Normalize to get probabilities
    probabilities = exp_z / np.sum(exp_z, axis=1, keepdims=True)

    return probabilities.squeeze() # Remove extra dimension if input was 1D

# --- Example Usage ---
# Works for small numbers
logits = np.array([2.0, 1.0, 0.1])
print(f"Softmax on small logits:\n{softmax(logits)}\n")

# Works for large numbers without overflow
logits_large = np.array([1000, 1010, 990])
print(f"Softmax on large logits:\n{softmax(logits_large)}")


In [ ]:
import numpy as np

logits = np.array([2.0, 1.0, 0.1])
print(logits.shape)

if logits.ndim == 1:
    logits = logits.reshape(1, -1) # (2D with 1 row, -1 for infer cols)

print(logits.shape)
# axis=1 -> across columns, per row. keepdims -> keeps shape (1,1) for broadcasting
max_logits = np.max(logits, axis=1, keepdims=True)
print(max_logits.shape)
print(max_logits.size)

In [ ]:
z = np.array([[2.0, 1.0, 0.1]])  # shape (1, 3)
print(np.max(z, axis=1).squeeze())         # max of the row

In [ ]:
# axis=1: across columns, i.e. per row (max in each row). axis=0: down rows, per column.
z = np.array([[2.0, 1.0, 0.1],
              [5.0, 3.0, 4.0]])  # shape (2, 3)
print(np.max(z, axis=1).squeeze())         # max of each row

In [ ]:
x = np.array([2.0, 1.0, 0.1])
print(x.shape)

softmax = np.exp(x) / np.exp(x).sum(axis=0, keepdims=True)
print(softmax)

In [ ]:
import torch

x = torch.tensor([2.0, 1.0, 0.1])  # shape (3,)

# softmax across classes; only valid axis for a 1D tensor is dim=0
softmax = torch.exp(x) / torch.exp(x).sum(dim=0, keepdim=True)
print(softmax)

In [ ]:
import torch

x = torch.tensor([[2.0, 1.0, 0.1],
                 [1.0, 3.0, 0.2]])  # shape (2, 3)

# softmax per row (per sample, across its classes) -> dim=1
softmax = torch.exp(x) / torch.exp(x).sum(dim=1, keepdim=True)
print(softmax)